# 04 — How a coding agent touches your files, then break it

You have already used a coding agent. Cursor, Claude Code, Copilot, a chat that "edits the repo." It feels as if the model opened the file and typed.

It did not. Module 00 said this: an LLM has no hands. It cannot list a folder, read a disk, or run a test. It can only emit a name and some JSON. **Your code** is what actually touches the files.

A coding agent is the official loop from module 03, plus a small set of file tools, pointed at a folder. That is the whole product. Today we build a tiny one, watch it fix a one-line bug, then break it on purpose.

This is the same idea as a more complete coding-agent lab you may have seen (skills, `map.md`, load-on-demand). Those extras are **context engineering** — module 05. Today is only the hands.


## 1. Learn

```mermaid
flowchart TD
    A["you type: the tests fail, fix them"] --> B["the model — no hands"]
    B -->|"emits tool_calls"| C["name: read_file<br/>arguments: {'path': 'pricing.py'}"]
    C --> D["your loop runs a Python function"]
    D --> E["the disk"]
    E -->|"you send back"| O["Observation"]
    O --> B
    B -->|"asks again"| G["write_file · run_tests · grep"]
    G --> D
    B -->|"stops, or we cap it"| Z["done"]
```

Why five functions, not one magic `do_code` tool? Because that is how those products work, underneath. Each tool is one kind of hand:

| Tool | The hand | Why it exists |
|---|---|---|
| `list_dir` | Look at the folder | The model cannot see the project. Someone has to name the files. |
| `read_file` | Open a file | Same as you clicking a tab. |
| `grep` | Search | So it does not have to read every file to find `apply_discount`. |
| `write_file` | Save | The only way a change lands on disk. |
| `run_tests` | Check the work | A claim that "it is fixed" is not evidence. |

There is **no** `run_bash`. A general shell is how an agent leaves the lesson and onto the machine. Module 10 is the sandbox. Today the only process we start is `unittest`, in this folder, with a timeout.

One more rule, from module 02: **your code decides**. Every path is checked against the workspace. A name like `../.env` is refused. The model asked. We said no.

Then we break it, so you see the failure before we name it later:

1. Two write tools that mean the same thing. Overlapping descriptions are how a router goes wrong (module 13).
2. A cap so short the job cannot finish. You stop the agent. That is not bureaucracy.

We use `MODEL_STRONG` for this module. Editing code is why that pin exists.


## 2. Do

### The files are already on disk

Look in `modules/04_coding_agent/workspace/`.

`pricing.py` — ten percent off a price. The sign is wrong. It **adds** the discount.

```
def apply_discount(price, percent):
    return price + price * percent / 100
```

For `price=100` and `percent=10` this returns **110**. A human wanted **90**.

`test_pricing.py` — one unit test that says so.

```
self.assertEqual(apply_discount(100, 10), 90)
```

No model wrote these. We did. The agent has not seen them yet.

### Load the environment and point at that folder


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = (
    os.environ.get("MODEL_STRONG", "").strip()
    or os.environ.get("MODEL_DEFAULT", "").strip()
)
assert api_key, "OPENAI_API_KEY is missing."
assert model, "Set MODEL_STRONG or MODEL_DEFAULT in .env."

client = OpenAI()
HERE = ROOT / "modules" / "04_coding_agent"
WORK = HERE / "workspace"
STARTER = HERE / "starter"

print("OPENAI_API_KEY is set:", True)
print("model for this module:", model)
print("workspace:", WORK)
print("files:", sorted(p.name for p in WORK.iterdir() if p.suffix == ".py"))


OPENAI_API_KEY is set: True
model for this module: gpt-5.4-mini
workspace: /Users/tarekatwan/Downloads/ai_agents_course/modules/04_coding_agent/workspace
files: ['pricing.py', 'test_pricing.py']


### The jail

Before any tool, one function: if the name would leave `workspace/`, refuse. Return a sentence, do not raise. The next thought can read a sentence.


In [2]:
def safe_path(name):
    if not name or name.strip() == "":
        return None, "need a file name"
    path = (WORK / name).resolve()
    work = WORK.resolve()
    if path != work and work not in path.parents:
        return None, "refused: path leaves the workspace"
    return path, ""


print("pricing.py ->", safe_path("pricing.py")[0].name)
print("../.env    ->", safe_path("../.env")[1])


pricing.py -> pricing.py
../.env    -> refused: path leaves the workspace


The second line is the point. Cursor will not (or should not) write your home directory because a model asked. Someone wrote a check like this.

### The five hands, as ordinary functions

Call them yourself first, the way we called `get_fact` before any schema. The model is not here yet.


In [3]:
def list_dir():
    names = sorted(p.name for p in WORK.iterdir() if p.is_file())
    return ", ".join(names) if names else "(empty)"


def read_file(path):
    target, err = safe_path(path)
    if err:
        return err
    if not target.is_file():
        return "file not found"
    return target.read_text()


print("list_dir:", list_dir())
print("--- pricing.py ---")
print(read_file("pricing.py"))


list_dir: pricing.py, test_pricing.py
--- pricing.py ---
def apply_discount(price, percent):
    return price + price * percent / 100



`grep` is search. `write_file` is save. `run_tests` starts **one** process: `python -m unittest` in this folder, 15 seconds, then it is killed. Not a shell. Not your laptop.


In [4]:
def grep(path, pattern):
    text = read_file(path)
    if text.startswith("refused:") or text == "file not found":
        return text
    hits = []
    for i, line in enumerate(text.splitlines(), start=1):
        if pattern in line:
            hits.append(str(i) + ": " + line)
    return "\n".join(hits) if hits else "no matches"


def write_file(path, content):
    target, err = safe_path(path)
    if err:
        return err
    target.write_text(content)
    return "wrote " + target.name + " (" + str(len(content)) + " chars)"


def run_tests():
    proc = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", str(WORK), "-q"],
        cwd=str(WORK),
        capture_output=True,
        text=True,
        timeout=15,
    )
    out = (proc.stdout or "") + (proc.stderr or "")
    return out.strip() if out.strip() else "(no output, exit " + str(proc.returncode) + ")"


print("grep discount:", grep("pricing.py", "percent"))
print("--- tests, no model ---")
print(run_tests())


grep discount: 1: def apply_discount(price, percent):
2:     return price + price * percent / 100
--- tests, no model ---
FAIL: test_ten_percent_off_100 (test_pricing.PricingTest.test_ten_percent_off_100)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/tarekatwan/Downloads/ai_agents_course/modules/04_coding_agent/workspace/test_pricing.py", line 7, in test_ten_percent_off_100
    self.assertEqual(apply_discount(100, 10), 90)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 110.0 != 90

----------------------------------------------------------------------
Ran 1 test in 0.000s

FAILED (failures=1)


The test fails. That is the ticket. A coding agent you use at work is looking at a fail string like this, then asking for `read_file`.

### Describe the hands to the model

Same as module 02. The schema is a description. It does not hook the function.


In [5]:
def schema(tool_name, description, **props):
    return {"type": "function", "function": {
        "name": tool_name, "description": description,
        "parameters": {"type": "object", "properties": props, "required": list(props)}}}

tools = [
    schema("list_dir", "List file names in the workspace."),
    schema("read_file", "Read one file. Argument is the file name only, not a path outside the workspace.", path={"type": "string"}),
    schema("grep", "Find a literal string in one workspace file.", path={"type": "string"}, pattern={"type": "string"}),
    schema("write_file", "Overwrite one workspace file with the given content.", path={"type": "string"}, content={"type": "string"}),
    schema("run_tests", "Run the workspace unit tests. No arguments."),
]
print([t["function"]["name"] for t in tools])


['list_dir', 'read_file', 'grep', 'write_file', 'run_tests']


### One create. Stop. Look.

The model has not touched the disk. If `finish_reason` is `tool_calls`, it is asking us to use a hand. We have not said yes yet.


In [6]:
SYSTEM = (
    "You fix Python in a small workspace. "
    "Use list_dir, read_file, grep, write_file, run_tests. "
    "Read the test and the code before you write. "
    "When the tests pass, stop and say what you changed."
)

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "The unit tests fail. Make them pass."},
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=400,
    reasoning_effort="none",
)
message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:      ", message.content)
print("n tool_calls: ", len(message.tool_calls or []))
if message.tool_calls:
    for call in message.tool_calls:
        print(" asked:", call.function.name, call.function.arguments[:160])


finish_reason: tool_calls
content:       None
n tool_calls:  1
 asked: list_dir {}


### Say yes to that one ask

Same dispatch as module 02. Unknown name → a sentence.


In [7]:
def run_coding_call(call):
    args = json.loads(call.function.arguments or "{}")
    name = call.function.name
    if name == "list_dir":
        return list_dir()
    if name == "read_file":
        return read_file(args.get("path", ""))
    if name == "grep":
        return grep(args.get("path", ""), args.get("pattern", ""))
    if name == "write_file":
        return write_file(args.get("path", ""), args.get("content", ""))
    if name == "run_tests":
        return run_tests()
    return "unknown tool"


messages.append(message)
for call in message.tool_calls or []:
    result = run_coding_call(call)
    print(call.function.name, "->")
    print(result[:500])
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})


list_dir ->
pricing.py, test_pricing.py


### Put the bug back, then run the loop

If the one-shot write already changed `pricing.py`, we copy the original from `starter/`. That is the only `write` we do by hand: restore a file we already showed you. Then the official loop, cap of 8.


In [8]:
shutil.copy(STARTER / "pricing_buggy.py", WORK / "pricing.py")
print(read_file("pricing.py"))


def apply_discount(price, percent):
    return price + price * percent / 100



In [9]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "The unit tests fail. Make them pass."},
]

for turn in range(8):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=400,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
    if not message.tool_calls:
        print(message.content)
        break
    messages.append(message)
    for call in message.tool_calls:
        result = run_coding_call(call)
        preview = result if len(result) < 240 else result[:240] + " ..."
        print(call.function.name, "->", preview.replace("\n", " / "))
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})


--- turn 1 finish_reason: tool_calls ---
list_dir -> pricing.py, test_pricing.py
run_tests -> ====================================================================== / FAIL: test_ten_percent_off_100 (test_pricing.PricingTest.test_ten_percent_off_100) / ---------------------------------------------------------------------- / Traceback (most ...


--- turn 2 finish_reason: tool_calls ---
read_file -> def apply_discount(price, percent): /     return price + price * percent / 100 / 
read_file -> import unittest / from pricing import apply_discount /  /  / class PricingTest(unittest.TestCase): /     def test_ten_percent_off_100(self): /         self.assertEqual(apply_discount(100, 10), 90) / 


--- turn 3 finish_reason: tool_calls ---
write_file -> wrote pricing.py (77 chars)


--- turn 4 finish_reason: tool_calls ---
run_tests -> ---------------------------------------------------------------------- / Ran 1 test in 0.000s /  / OK


--- turn 5 finish_reason: stop ---
Fixed `apply_discount` in `pricing.py` so it subtracts the discount instead of adding it.

Tests now pass.


## 3. Observe

Do not trust the last sentence. Read the file. Run the tests. The disk is the ground truth. That is also how you should treat Cursor.


In [10]:
print(read_file("pricing.py"))
print("--- tests now ---")
print(run_tests())


def apply_discount(price, percent):
    return price - price * percent / 100

--- tests now ---
----------------------------------------------------------------------
Ran 1 test in 0.000s

OK


If the sign is minus and the tests say `OK`, the loop did what the product demo promised. If the model said "fixed" and the tests still fail, believe the tests.

### Break 1 — two tools that mean the same thing

Coding agents fail in public when two tools overlap ("edit" vs "apply_patch" vs "write"). We add `save_file` with a description that is basically `write_file`. Restore the bug, run a short loop, list the names it picked.


In [11]:
def save_file(path, content):
    return write_file(path, content)


tools_overlap = tools + [schema("save_file", "Write or update a file in the workspace.", path={"type": "string"}, content={"type": "string"})]


def run_coding_call_overlap(call):
    if call.function.name == "save_file":
        args = json.loads(call.function.arguments or "{}")
        return save_file(args.get("path", ""), args.get("content", ""))
    return run_coding_call(call)


shutil.copy(STARTER / "pricing_buggy.py", WORK / "pricing.py")
messages = [
    {"role": "system", "content": SYSTEM + " You may write files with write_file or save_file."},
    {"role": "user", "content": "The unit tests fail. Make them pass."},
]
picked = []
for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools_overlap,
        max_completion_tokens=400,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    if not message.tool_calls:
        print("final:", message.content)
        break
    messages.append(message)
    for call in message.tool_calls:
        picked.append(call.function.name)
        result = run_coding_call_overlap(call)
        print("turn", turn + 1, call.function.name)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

print("tools it picked:", picked)


turn 1 list_dir
turn 1 run_tests


turn 2 read_file
turn 2 read_file


turn 3 write_file


turn 4 run_tests


final: Fixed `apply_discount` in `pricing.py` to subtract the discount instead of adding it. Tests now pass.
tools it picked: ['list_dir', 'run_tests', 'read_file', 'read_file', 'write_file', 'run_tests']


If both `write_file` and `save_file` appear, that was not a real choice. Module 13 measures this. Module 14 reminds you: your code still ran whichever name it picked.

### Break 2 — you stop it

Same job. Cap of 2. It may list and read, and then we halt. A loop with no cap is how a key goes to zero. We use 2, not infinity, because this is a classroom. The sentence is the same.


In [12]:
shutil.copy(STARTER / "pricing_buggy.py", WORK / "pricing.py")
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "The unit tests fail. Make them pass."},
]
for turn in range(2):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=400,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
    if not message.tool_calls:
        print(message.content)
        break
    messages.append(message)
    for call in message.tool_calls:
        result = run_coding_call(call)
        print(call.function.name)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
else:
    print("stopped: cap was 2. The tests were not the stop condition. We were.")

print("--- tests after the short cap ---")
print(run_tests())


--- turn 1 finish_reason: tool_calls ---
list_dir
run_tests


--- turn 2 finish_reason: tool_calls ---
read_file
read_file
stopped: cap was 2. The tests were not the stop condition. We were.
--- tests after the short cap ---
FAIL: test_ten_percent_off_100 (test_pricing.PricingTest.test_ten_percent_off_100)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/tarekatwan/Downloads/ai_agents_course/modules/04_coding_agent/workspace/test_pricing.py", line 7, in test_ten_percent_off_100
    self.assertEqual(apply_discount(100, 10), 90)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 110.0 != 90

----------------------------------------------------------------------
Ran 1 test in 0.000s

FAILED (failures=1)


## 4. Challenge

A second file, already written, sitting in `starter/`. We copy it in. `line_total` **adds** quantity and price. The test wants `2 * 10 = 20`.

Use the **good** tool list (no `save_file`) and a cap of 8. Do not edit the file by hand.

Bind `n_lookups` (how many tool results you appended) and `tests_after` (what `run_tests()` returns). The check looks for a tool having run, and for `OK` in the test output.


In [13]:
shutil.copy(STARTER / "pricing_fixed.py", WORK / "pricing.py")
shutil.copy(STARTER / "totals_buggy.py", WORK / "totals.py")
shutil.copy(STARTER / "test_totals.py", WORK / "test_totals.py")
print(list_dir())
print(read_file("totals.py"))
print(run_tests())


pricing.py, test_pricing.py, test_totals.py, totals.py
def line_total(qty, price):
    return qty + price

FAIL: test_two_at_ten (test_totals.TotalsTest.test_two_at_ten)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/tarekatwan/Downloads/ai_agents_course/modules/04_coding_agent/workspace/test_totals.py", line 7, in test_two_at_ten
    self.assertEqual(line_total(2, 10), 20)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 12 != 20

----------------------------------------------------------------------
Ran 2 tests in 0.000s

FAILED (failures=1)


In [ ]:
# n_lookups, tests_after = ...


In [ ]:
assert n_lookups >= 1, "the loop should have run at least one tool"
assert "OK" in tests_after, "tests_after should be the passing unittest output"
print("n_lookups:", n_lookups)
print(tests_after)
